In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 加载model,tokenizer
custom_model_name = "fine-tuned-models/nlslb-200-1.3B/zh2ko_1107"

model = AutoModelForSeq2SeqLM.from_pretrained(custom_model_name)
tokenizer = AutoTokenizer.from_pretrained(custom_model_name)

In [2]:
from transformers import pipeline

translator = pipeline("translation", model=model, tokenizer=tokenizer,
                      src_lang=tokenizer.src_lang,
                      tgt_lang=tokenizer.tgt_lang,
                      device="cuda:0", max_length=400, return_tensors=True)

In [3]:
import pandas as pd

df = pd.read_excel("not_fanyi_Lang_korea_韩文_preprocessed.xlsx")

source = df["zh-CN"].to_list()  # 待翻译的句子

In [4]:
from tqdm.notebook import tqdm

# 进行翻译并提取结果
translated = [translator(item)[0] for item in tqdm(source)]
translated_token_ids = [item['translation_token_ids'].tolist() for item in translated]
translated_text = tokenizer.batch_decode(translated_token_ids, skip_special_tokens=False)

  0%|          | 0/796 [00:00<?, ?it/s]

D:\LongtuKoreaTranslationModel\venv\lib\site-packages\transformers\pipelines\base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
Your input_length: 499 is bigger than 0.9 * max_length: 400. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Your input_length: 405 is bigger than 0.9 * max_length: 400. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)


In [5]:
df_compare = pd.DataFrame({
    "source": source,
    "candidates": translated_text,
})
df_compare

,source,candidates
0,07时59分,</s>kor_Hang 07时59分</s>
1,<start>魔音<middle>마음<end>乱心,</s>kor_Hang<start> 마음<end> 이 안심</s>
2,“零”，“一”，“二”，“三”，“四”，“五”，“六”，“七”，“八”，“九”，“十”，“十...,"</s>kor_Hang 떡이 작은 떡, 떡이 하나, 떡이 두, 떡이 세, 떡이 네,..."
3,<start>服饰<middle>장신구<end>·狰骨怒涛,</s>kor_Hang<start> 장신구<end> -]골노정</s>
4,<start>操作<middle>조작<end>太频繁，请<code_id=0>1<code...,</s>kor_Hang<start> 조작<end> 빈도가 너무 높습니다. <code...
...,...,...
791,2888#r元#r礼#r包,</s>kor_Hang 2888#r원 #r#礼#r包</s>
792,"请勿相信任何声称<start>自己<middle>자신<end>认识官方人员,然后要求线下转...",</s>kor_Hang<start> 자신<end> 만의 <start> 알 수 있는 ...
793,<start>区服<middle>서버<end>列表载<start>失败<middle>실패...,</s>kor_Hang<start> 서버<end> 로드 <start> 실패<end>...
794,<start>个人<middle>개인<end><start>竞技场<middle>아레나<...,</s>kor_Hang<start> 개인<end> <start> 아레나<end></s>


In [6]:
file_name = "translation_result_of_{0}".format(custom_model_name.replace("/", "_"))
df_compare.to_excel("{0}.xlsx".format(file_name), index=False)
df_compare.to_csv("{0}.csv".format(file_name), index=False)